## Importing libraries

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as st

## Loading the Datasets

In [2]:
info = pd.read_csv("./AB_testing/Data/Clean_data/df_final_demo_cleaned.csv")
exp = pd.read_pickle("./AB_testing/Data/Clean_data/final_test_data.pkl")
version = pd.read_csv("./AB_testing/Data/Clean_data/df_final_experiment_clients_cleaned.csv")

In [3]:
info.head()

,client_id,tenure_years,tenure_months,age,gender,num_accts,balance,calls_last_6_months,logons_last_6_months
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0


In [4]:
info.shape

(70609, 9)

In [5]:
exp.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


In [6]:
version.head()

,client_id,group_test
0,9988021,test
1,8320017,test
2,4033851,control
3,1982004,test
4,9294070,control


In [7]:
version.shape

(70609, 2)

## Merging Datasets

In [8]:
info_com = pd.merge(info, version, on = "client_id")

In [9]:
info_com.head()

,client_id,tenure_years,tenure_months,age,gender,num_accts,balance,calls_last_6_months,logons_last_6_months,group_test
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,test
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,control
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,test
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,test
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,control


In [10]:
info_com["group_test"].isna().sum()

np.int64(20109)

In [11]:
info_com.dropna(subset=["group_test"], inplace = True)

In [12]:
info_com.shape

(50500, 10)

## Hypothesis 4

For the 4th hypothesis we want if there is more completion depending on the age:
- H0: test group completion by age <= control group completion by age
- H1: test group completion by age > control group completion by age

### Preparation H4

In [13]:
mixed = pd.merge(exp, info_com, on = "client_id")

# now we have a huge dataset. We'll clean it and leave it for clarity

In [14]:
mixed.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'source', 'group_test_x', 'tenure_years', 'tenure_months', 'age',
       'gender', 'num_accts', 'balance', 'calls_last_6_months',
       'logons_last_6_months', 'group_test_y'],
      dtype='str')

In [15]:
clean_mixed = mixed.drop(columns = ['visitor_id', 'visit_id', 'date_time',
       'source', 'tenure_years', 'tenure_months',
       'gender', 'num_accts', 'balance', 'calls_last_6_months',
       'logons_last_6_months', 'group_test_y'])

In [16]:
clean_mixed.head()

,client_id,process_step,group_test_x,age
0,3561384,confirm,test,59.5
1,3561384,confirm,test,59.5
2,7338123,start,test,23.5
3,7338123,step_1,test,23.5
4,7338123,step_2,test,23.5


In [17]:
h4 = clean_mixed[clean_mixed['process_step']=='confirm']

# we filter in order to have only people who reached confirm in the process

In [18]:
h4.head()

,client_id,process_step,group_test_x,age
0,3561384,confirm,test,59.5
1,3561384,confirm,test,59.5
12,7338123,confirm,test,23.5
17,2478628,confirm,test,47.0
35,3479519,confirm,control,52.5


In [19]:
h4 = h4.rename(columns = {"group_test_x" : "group_test"})

In [20]:
h4.shape

(42936, 4)

In [21]:
h4.isna().sum()

client_id        0
process_step     0
group_test       0
age             13
dtype: int64

In [22]:
h4["age"] = h4["age"].fillna(h4["age"].mean())

# we have 13 null values so we are filling with age mean

In [23]:
h4["group_test"].value_counts()

group_test
test       25600
control    17336
Name: count, dtype: int64

In [24]:
h4_control = h4[h4["group_test"] == "control"]["age"]
h4_test = h4[h4["group_test"] == "test"]["age"]

### Testing H4

In this case, we have two categorical variables, so we'll run a t-test. We'll use a significance level of 5%. As we said, our hypothesis is:

- H0: test group completion by age <= control group completion by age
- H1: test group completion by age > control group completion by age

In [25]:
st.ttest_ind(h4_test, h4_control, alternative = "greater")

TtestResult(statistic=np.float64(1.6943009696681348), pvalue=np.float64(0.04510767954781082), df=np.float64(42934.0))

As the p_value is lower than 0.05, we can reject our null hypothesis and say that there in general, the test group has been more successful across different ages.

## Hypothesis 5

For the 5th hypothesis we want if there is more completion depending on the gender:
- H0: test group completion by gender = control group completion by gender
- H1: test group completion by gender != control group completion by gender

### Preparation H5

In [26]:
mixed2 = pd.merge(exp, info_com, on = "client_id")

In [27]:
mixed2.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'source', 'group_test_x', 'tenure_years', 'tenure_months', 'age',
       'gender', 'num_accts', 'balance', 'calls_last_6_months',
       'logons_last_6_months', 'group_test_y'],
      dtype='str')

In [28]:
clean_mixed2 = mixed.drop(columns = ['visitor_id', 'visit_id', 'date_time',
       'source', 'tenure_years', 'tenure_months',
       'age', 'num_accts', 'balance', 'calls_last_6_months',
       'logons_last_6_months', 'group_test_y'])

In [29]:
clean_mixed2.head()

,client_id,process_step,group_test_x,gender
0,3561384,confirm,test,U
1,3561384,confirm,test,U
2,7338123,start,test,M
3,7338123,step_1,test,M
4,7338123,step_2,test,M


In [40]:
h5 = clean_mixed2[clean_mixed2['process_step']=='confirm']

# we filter in order to have only people who reached confirm in the process

In [41]:
h5 = h5[(h5['gender']=='M') | (h5['gender']=='F')]

# we filter in order to remove the unknown gender

In [43]:
h5 = h5.rename(columns = {"group_test_x" : "group_test"})

In [44]:
h5.head()

,client_id,process_step,group_test,gender
12,7338123,confirm,test,M
17,2478628,confirm,test,F
35,3479519,confirm,control,F
42,5477656,confirm,control,M
51,8631696,confirm,test,M


In [50]:
contingency_gender = pd.crosstab(h5["group_test"], h5["gender"])
contingency_gender

gender,F,M
group_test,,
control,5465,5988
test,8151,8922


### Testing H5

In this case, we have two categorical variables, so we'll run a Chi2 test. We'll use a significance level of 5%. As we said, our hypothesis is:

- H0: test group completion by gender = control group completion by gender
- H1: test group completion by gender != control group completion by gender

In [48]:
st.chi2_contingency(contingency_gender)

Chi2ContingencyResult(statistic=np.float64(0.0008901158402949607), pvalue=np.float64(0.976198797384497), dof=1, expected_freq=array([[5466.73378672, 5986.26621328],
       [8149.26621328, 8923.73378672]]))

There is no evidence to reject our null hypothesis, meaning that there is no difference in completion in the groups and gender